In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('data/health_v1.csv')
df.head()

,ID;Weight;Smoking;Exercise;Cholesterol;Income;Happiness;BirthYear;Sex
1;75;0;3;5,8;3230;59;1964;1
2;70;0;6;5,9;1420;43;1952;1
3;76;0;6;6;4930;74;1951;1,NaN
4;62;0;8;4,7;2970;58;1959;1
5;44;0;8;4,3;4780;64;1925;1


Initial loading attempt fails as the file is not in correct format; Common errors in data preprocessing.

It is required to handle these errors before proceeding.

One problem can be fixed immediately by adding a sep parameter as all the values go into first column due to values being separated by semicolons, not commas.

In [4]:
df = pd.read_csv('data/health_v1.csv', sep=';', na_values='NaN')
df.head()

,ID,Weight,Smoking,Exercise,Cholesterol,Income,Happiness,BirthYear,Sex
0,1,75.0,0.0,3,"5,8",3230,59.0,1964,1
1,2,70.0,0.0,6,"5,9",1420,43.0,1952,1
2,3,76.0,0.0,6,6,4930,74.0,1951,1
3,4,62.0,0.0,8,"4,7",2970,58.0,1959,1
4,5,44.0,0.0,8,"4,3",4780,64.0,1925,1


While things look better, the values in Cholesterol column raise some suspicion. The decimal separator is a comma, which is not the default for pandas. This can be fixed as well:

In [6]:
df = pd.read_csv('data/health_v1.csv', sep=';', na_values='NaN', decimal=',')
df.head()

,ID,Weight,Smoking,Exercise,Cholesterol,Income,Happiness,BirthYear,Sex
0,1,75.0,0.0,3,5.8,3230,59.0,1964,1
1,2,70.0,0.0,6,5.9,1420,43.0,1952,1
2,3,76.0,0.0,6,6.0,4930,74.0,1951,1
3,4,62.0,0.0,8,4.7,2970,58.0,1959,1
4,5,44.0,0.0,8,4.3,4780,64.0,1925,1


While things look much better now, there are still some suspicion about the data types.

In the original dataset, both the **Weight** and **Exercise** fields seemed to have integer values, but now the former seems to have been transformed into a float.

You can check the datatypes of the columns with the dtypes attribute:

In [7]:
df.dtypes

ID               int64
Weight         float64
Smoking        float64
Exercise           str
Cholesterol    float64
Income             str
Happiness      float64
BirthYear          str
Sex                str
dtype: object

In [8]:
df.Exercise.unique()

<ArrowStringArray>
['3', '6', '8', '7', '5', '4', '2', '9', '10', '0', '1', ' ']
Length: 12, dtype: str

**unique()** method reveals that there is an extra value, a space, in the Exercise column.

You can fix this by adding the **na_values** parameter to the **read_csv** function.

In [10]:
df = pd.read_csv('data/health_v1.csv', sep=';', na_values=['', ' '], decimal=',')
df.Exercise.unique()

array([ 3.,  6.,  8.,  7.,  5.,  4.,  2.,  9., 10.,  0.,  1., nan])

In [11]:
df.dtypes

ID               int64
Weight         float64
Smoking        float64
Exercise       float64
Cholesterol    float64
Income         float64
Happiness      float64
BirthYear      float64
Sex                str
dtype: object

Now the issue is mostly fixed except for the **Sex** variable as it's still a String. You can convert it to a category:

In [12]:
df['Sex'] = pd.Categorical(df['Sex'])

And check the unique values once more:

In [13]:
df.Sex.unique()

['1', '2', 'Male', 'Female']
Categories (4, str): ['1', '2', 'Female', 'Male']

There appears to be four different values:
- 1,
- 2,
- Male,
- Female.

The Sex column has initially been encoded so that **1** represents male and **2** represents female.

The written values **Male** and **Female** are most likely errors in the dataset, so you could replace them with missing values.

In [14]:
df['Sex'] = df['Sex'].astype(str)
df['Sex'] = df['Sex'].replace({'Male': '1', 'Female': '2'})
df['Sex'] = df['Sex'].astype('category')

In [15]:
df.describe(include='all')

,ID,Weight,Smoking,Exercise,Cholesterol,Income,Happiness,BirthYear,Sex
count,1000.000000,999.000000,999.000000,998.000000,1000.00000,997.000000,999.000000,999.000000,1000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,523
mean,500.500000,79.717718,0.232232,5.092184,5.72750,3069.829488,51.171171,1954.111111,NaN
std,288.819436,25.252719,0.422468,2.030736,1.39649,1264.497563,17.511569,21.443328,NaN
min,1.000000,8.000000,0.000000,0.000000,-7.50000,0.000000,0.000000,1879.000000,NaN
25%,250.750000,70.000000,0.000000,4.000000,4.87500,2220.000000,39.000000,1939.000000,NaN
50%,500.500000,80.000000,0.000000,5.000000,5.80000,3140.000000,50.000000,1954.000000,NaN
75%,750.250000,88.000000,0.000000,6.000000,6.60000,3890.000000,63.000000,1969.000000,NaN
